# VANDAlize — 02: Analysis

Reproduces every figure and table in the paper from pre-extracted activations.
**Runs on CPU in ~2 minutes.** No GPU or model weights required.

Pipeline:
1. Load activations + labels per condition
2. PCA at layer 14 (visual separation of harmful vs benign)
3. Layer sweep (difference-of-means probe, train/test, all layers)
4. **Cross-condition transfer matrix** (accuracy + AUROC) — the headline experiment
5. Harm directions and pairwise cosine similarity
6. Magnitude (signal-strength) comparison
7. Bootstrap CIs on cosines

Convention: label 1 = harmful, 0 = benign. Harm direction = mean(harmful) - mean(benign).

To switch models (LLaMA-2-13B, SEA-LION v1, SEA-LION v3) change the file paths in
the `CONDITIONS` dict below — the analysis code is model-agnostic.

## 1. Setup

In [ ]:
!pip install -q torch numpy scikit-learn matplotlib seaborn

In [ ]:
import sys, os
sys.path.insert(0, "../src")   # so we can import the shared module
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from vandalize import (
    DiffOfMeansProbe, harm_direction_norm, layer_sweep,
    cosine_matrix, transfer_matrix, bootstrap_cosine, SEED,
)

np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "../data"        # in Colab, upload .pt files and set this to "/content"
LAYER = 14                  # probe layer used throughout the paper

## 2. Load activations and labels

Each `.pt` activation file is a dict `{"<condition> run <k>": {layer_idx: tensor[n, hidden]}}`.
Each label file is `{"<condition> run <k>": tensor[n]}`. Edit `CONDITIONS` to point at
the files for the model you are analyzing. The example below is LLaMA-2-13B.

In [ ]:
# condition -> (activations_file, labels_file, inner_key)
CONDITIONS = {
    "english":  ("English_1_activations_all_layers.pt",        "English_1_nat_labels.pt",        "english run 1"),
    "filipino": ("filipino_1_activations_all_layers.pt",       "filipino_1_nat_labels.pt",       "filipino run 1"),
    "taglish":  ("taglish_1_activations_all_layers.pt",        "taglish_1_nat_labels.pt",        "taglish run 1"),
}

acts, labels = {}, {}
for cond, (a_file, l_file, key) in CONDITIONS.items():
    a = torch.load(f"{DATA_DIR}/{a_file}", map_location="cpu", weights_only=False)[key]
    l = torch.load(f"{DATA_DIR}/{l_file}", map_location="cpu", weights_only=False)[key]
    acts[cond]   = {L: a[L].numpy() for L in a}     # {layer: array[n, hidden]}
    labels[cond] = l.numpy()

CONDS = list(CONDITIONS.keys())
for c in CONDS:
    y = labels[c]
    n_layers = len(acts[c]); n, h = acts[c][0].shape
    print(f"{c:10s} layers={n_layers} n={n} hidden={h} harmful={(y==1).sum()} benign={(y==0).sum()}")

## 3. PCA at layer 14

Unsupervised view of how separable harmful (red) and benign (blue) prompts are
in each condition. Cleaner separation in English than in Taglish is the visual
intuition behind the quantitative results.

In [ ]:
fig, axes = plt.subplots(1, len(CONDS), figsize=(6*len(CONDS), 5))
if len(CONDS) == 1: axes = [axes]
for ax, cond in zip(axes, CONDS):
    X = acts[cond][LAYER]; y = labels[cond]
    pca = PCA(n_components=2, random_state=SEED).fit(X)
    Z = pca.transform(X)
    for cls, color, name in [(1,"red","harmful"), (0,"blue","benign")]:
        m = y == cls
        ax.scatter(Z[m,0], Z[m,1], c=color, alpha=0.6, s=20, label=name)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_title(f"{cond} (layer {LAYER})"); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle(f"PCA of layer {LAYER} representations", y=1.02)
plt.tight_layout(); plt.savefig("../figures/pca.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Layer sweep

Train a difference-of-means probe at every layer (80/20 split). Confirms a harm
direction exists in every condition (positive control) and shows where it peaks.

In [ ]:
fig, axes = plt.subplots(1, len(CONDS), figsize=(6*len(CONDS), 5), sharey=True)
if len(CONDS) == 1: axes = [axes]
for ax, cond in zip(axes, CONDS):
    L, tr, te = layer_sweep(acts[cond], labels[cond])
    ax.plot(L, tr, label="Train", color="C0")
    ax.plot(L, te, label="Test",  color="C1")
    ax.axhline(0.5, ls="--", color="gray", alpha=0.6)
    ax.set_title(cond); ax.set_xlabel("Layer"); ax.set_ylabel("Accuracy")
    ax.legend(); ax.grid(alpha=0.3)
    print(f"{cond:10s} peak test={te.max():.3f} @ layer {L[te.argmax()]}, layer {LAYER} test={te[LAYER]:.3f}")
plt.suptitle("Layer sweep: difference-of-means probe", y=1.02)
plt.tight_layout(); plt.savefig("../figures/layer_sweep.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Cross-condition transfer matrix (headline)

Train a probe on condition X (rows), evaluate on condition Y (columns).
**Accuracy** depends on the decision threshold; **AUROC** ignores it (pure ranking).

The key signature: AUROC stays high across all cells while accuracy drops in
cross-condition cells. That means the *ranking* of harmful-vs-benign transfers,
but the *decision threshold* does not — consistent with magnitude attenuation
(Section 6) rather than a wrong or missing harm direction.

In [ ]:
M_acc, M_auc = transfer_matrix(acts, labels, CONDS, layer=LAYER)

fig, axes = plt.subplots(1, 2, figsize=(7*len(CONDS)/1.5, 5))
for ax, M, title in [(axes[0], M_acc, "Accuracy"), (axes[1], M_auc, "AUROC")]:
    sns.heatmap(M, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=1.0,
                xticklabels=CONDS, yticklabels=CONDS, ax=ax, cbar_kws={"label": title})
    ax.set_xlabel("Test condition"); ax.set_ylabel("Train condition")
    ax.set_title(f"Cross-condition transfer ({title}, layer {LAYER})")
plt.tight_layout(); plt.savefig("../figures/transfer_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

print("Within-condition (diagonal) vs cross-condition (off-diagonal) accuracy:")
for i, c in enumerate(CONDS):
    off = np.mean([M_acc[i,j] for j in range(len(CONDS)) if j != i])
    print(f"  train={c:10s} within={M_acc[i,i]:.3f} cross_mean={off:.3f} drop={M_acc[i,i]-off:.3f}")

## 6. Harm directions, cosine similarity, and magnitude

Cosine: are the harm directions pointing the same way across conditions?
Magnitude (norm of the raw difference of means): how strong is the harm signal?

The magnitude asymmetry — English harm direction ~1.5x stronger than Filipino/Taglish —
is the core quantitative finding, and it recurs across models.

In [ ]:
C, directions = cosine_matrix(acts, labels, CONDS, layer=LAYER)

fig, ax = plt.subplots(figsize=(1.6*len(CONDS)+2, 1.4*len(CONDS)+2))
sns.heatmap(C, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1,
            xticklabels=CONDS, yticklabels=CONDS, ax=ax, cbar_kws={"label":"cosine"})
ax.set_title(f"Pairwise cosine similarity of harm directions (layer {LAYER})")
plt.tight_layout(); plt.savefig("../figures/cosine_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

print("Harm direction norms (signal strength), layer", LAYER)
norms = {c: harm_direction_norm(acts[c][LAYER], labels[c]) for c in CONDS}
for c in CONDS:
    print(f"  {c:10s} {norms[c]:.4f}")
if "english" in norms:
    for c in CONDS:
        if c != "english":
            print(f"  english / {c} ratio = {norms['english']/norms[c]:.3f}")

# Magnitude bar chart
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(CONDS, [norms[c] for c in CONDS], color=["#1f77b4","#2ca02c","#ff7f0e"][:len(CONDS)])
ax.set_ylabel("Harm direction norm"); ax.set_title(f"Harm signal strength by condition (layer {LAYER})")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("../figures/magnitude.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Bootstrap CIs on cosines

With small n per condition, the direction estimates are noisy. Bootstrap resampling
tells you which cosine differences are real vs sampling noise.

In [ ]:
pairs = [(CONDS[i], CONDS[j]) for i in range(len(CONDS)) for j in range(i+1, len(CONDS))]
print("Bootstrap 95% CIs on pairwise cosines (layer {}, 1000 resamples):".format(LAYER))
for a, b in pairs:
    point, lo, hi, _ = bootstrap_cosine(acts, labels, a, b, layer=LAYER, n_boot=1000)
    print(f"  {a:10s} vs {b:10s}: {point:.4f}  [{lo:.4f}, {hi:.4f}]")

---
### Switching models

To reproduce the SEA-LION v1 or v3 results, change the `CONDITIONS` dict in
Section 2 to point at the corresponding `.pt` files (e.g. `SEAv3English_1_...`),
then re-run from Section 2. All analysis code is model-agnostic.

For SEA-LION v1 the probe is near chance at layer 14 (no usable linear harm
direction); for SEA-LION v3 the English/Tagalog magnitude asymmetry replicates
the LLaMA-2-13B pattern. See the paper's model-comparison table.